# Final-Layer DP Usage for Pauli-Path ZNE

This notebook shows the current workflow for the newest final-layer dynamic-programming (DP) code:

1. Sample prefixes with either the **gate-wise triggered sampler** or the **full-layer DP sampler**.
2. Replace explicit sampling of the last circuit layer with exact final-layer DP enumeration of all nonzero terminal Pauli paths.
3. Estimate the sector functions

$$Q^\pm(s)=\frac{\sum_{\gamma\in \pm}|A_\gamma|D_\gamma^s}{\sum_{\gamma\in \pm}|A_\gamma|}.$$

4. Compare pooled, group-mean, and median-of-means (MoM) summaries.
5. Compute an MPS reference curve for the noisy observable $O(s)$ using the same independent/product noise model.

**Important:** do not use `legacy_sum` noise unless specifically testing old behavior. The current sampling comparisons use independent/product Pauli noise.

## File Map

Core files for the final-layer DP workflow:

- `Pauli_path_Heis_full_layer_sampling_restricted.py`: full-layer sampler and final-layer DP terminal routines, including `terminal_all_good_qpm_grid_sums`.
- `Pauli_path_Heis_mixture_trigger.py`: gate-wise triggered prefix sampler.
- `benchmark_q_curve_convergence.py`: exposes `gate_q_curve_aggregate` and `dp_q_curve_aggregate`, which compute final-DP $Q^\pm(s)$ curves for both methods.
- `final_dp_mom_convergence_sweep.py`: production-style pooled/group-mean/MoM convergence sweep.
- `pauli_mps_solver.py`: MPS reference simulator.
- `compute_mps_zizj_vs_s.py` and `sweep_mps_zizj_chi_vs_s.py`: command-line MPS drivers.

Useful diagnostics:

- `diagnose_q_batch_heavytails.py`: checks denominator concentration and whether high-denominator groups pull pooled estimates away from MoM.
- `compare_final_dp_vs_single_pauli_q.py`: compares final-layer DP against the old single-final-Pauli terminal sampler.
- `sweep_final_dp_vs_single_convergence.py`: sample-count convergence comparison for final-DP versus single-final-Pauli terminal sampling.

In [ ]:
from pathlib import Path
import sys
import csv
import time

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pauli_mps_solver.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np

import Pauli_path_Heis_mixture_trigger as gate
import Pauli_path_Heis_full_layer_sampling_restricted as layer
from benchmark_q_curve_convergence import (
    dp_q_curve_aggregate,
    find_s1_index,
    gate_q_curve_aggregate,
    parse_s_grid,
    product_eta_from_lambda,
    row_from_aggregate,
)
from final_dp_mom_convergence_sweep import summarize as summarize_q_groups
from pauli_mps_solver import evolve_observable_backward_mps, pauli_zz

plt.rcParams.update({"figure.dpi": 130})

## Shared Parameters

The default values below are intentionally small enough for a notebook sanity check. Increase `N_QUBITS`, `N_STEPS`, `TOTAL_SAMPLES`, and `CHI_MAX` for production runs.

In [ ]:
N_QUBITS = 20
N_STEPS = 6
Q1, Q2 = 6, 10
PHI = 0.2
LAMBDA_BASE = 1.0e-3
S_GRID = np.array([1.0, 2.0, 4.0, 8.0], dtype=np.float64)

# MoM/convergence settings for the notebook demo.
# Production runs usually use more total samples and still keep 8-16 groups.
TOTAL_SAMPLES = [10_000, 100_000, 1_000_000]
N_GROUPS = 16

# MPS settings. For larger/deeper systems use chi=350 or higher and check chi convergence.
CHI_MAX = 128
CUTOFF = 1.0e-10
SVD_METHOD = "auto"

## MPS Reference Observable Curve

The MPS reference computes the noisy observable

$$O(s)=\langle Z_i Z_j\rangle_{\lambda=s\lambda_0}.$$

For consistency with the current sampling noise model, use:

- `noise_model="independent"`
- `noise_placement="layer"`

This is an observable reference curve, not a direct estimate of $Q^\pm(s)$.

In [ ]:
def run_mps_curve(
    n_qubits=N_QUBITS,
    n_steps=N_STEPS,
    q1=Q1,
    q2=Q2,
    phi=PHI,
    lambda_base=LAMBDA_BASE,
    s_values=(0.0, 1.0, 2.0, 4.0, 8.0),
    chi_max=CHI_MAX,
):
    target = pauli_zz(n_qubits, q1, q2)
    rows = []
    for s in s_values:
        lam = np.full((n_qubits, 3), lambda_base * s, dtype=np.float64)
        t0 = time.perf_counter()
        value, _, info = evolve_observable_backward_mps(
            target,
            n_qubits=n_qubits,
            phi=phi,
            lam_xyz=lam,
            n_steps=n_steps,
            chi_max=chi_max,
            cutoff=CUTOFF,
            svd_method=SVD_METHOD,
            noise_model="independent",
            noise_placement="layer",
            use_lightcone=True,
            return_mps=True,
        )
        rows.append(
            {
                "s": float(s),
                "O_mps": float(value),
                "runtime_s": time.perf_counter() - t0,
                "max_bond": int(np.max(info["bond_dims"])),
                "max_discarded_weight": float(np.max(info["discarded_by_backward_step"])),
            }
        )
    return rows


mps_rows = run_mps_curve()
mps_rows

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot([r["s"] for r in mps_rows], [r["O_mps"] for r in mps_rows], marker="o")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_xlabel("s")
ax.set_ylabel(r"MPS $O(s)$")
ax.set_title(rf"MPS reference: $Z_{{{Q1}}}Z_{{{Q2}}}$, N={N_QUBITS}, L={N_STEPS}")
ax.grid(True, alpha=0.25)

## Final-Layer DP $Q^\pm(s)$ Sampling

For each sampled prefix $P_{L-1}$, the final-layer DP exactly computes terminal sector sums:

$$T^\pm_{\rm num}(P_{L-1},s),\qquad T^\pm_{\rm den}(P_{L-1}).$$

The remaining Monte Carlo estimator is over prefixes:

$$Q^\pm(s)=\frac{\mathbb{E}[W(P_{L-1})T^\pm_{\rm num}(P_{L-1},s)]}{\mathbb{E}[W(P_{L-1})T^\pm_{\rm den}(P_{L-1})]}.$$

The DP is exact conditional on the prefix. Any remaining rare-event issue is prefix-sampling variance, not final-DP bias.

In [ ]:
def group_size_for(total_samples, n_groups):
    return max(1, int(np.ceil(total_samples / n_groups)))


def run_final_dp_q_groups(
    total_samples_list=TOTAL_SAMPLES,
    n_groups=N_GROUPS,
    methods=("gate", "layer"),
    n_qubits=N_QUBITS,
    n_steps=N_STEPS,
    q1=Q1,
    q2=Q2,
    phi=PHI,
    lambda_base=LAMBDA_BASE,
    s_grid=S_GRID,
):
    s1_idx = find_s1_index(s_grid)
    init_pauli = np.zeros(n_qubits, dtype=np.int8)
    init_pauli[q1] = 3
    init_pauli[q2] = 3

    lam_xyz = np.full((n_qubits, 3), lambda_base, dtype=np.float64)
    eta_xyz = product_eta_from_lambda(lam_xyz)

    even_gates, odd_gates = gate.make_even_odd_layers(n_qubits)
    trans_g, probs_g, is_comm_g, sign_g, amp_factor_g = gate.build_transition_tables(phi)
    trans_l, signed_l, abs_l, n_branches_l, *_ = layer.build_full_layer_tables(phi)

    group_rows = []
    for requested_total in total_samples_list:
        group_size = group_size_for(requested_total, n_groups)
        actual_total = group_size * n_groups
        for group in range(1, n_groups + 1):
            if "gate" in methods:
                t0 = time.perf_counter()
                agg = gate_q_curve_aggregate(
                    init_pauli,
                    even_gates,
                    odd_gates,
                    trans_g,
                    probs_g,
                    is_comm_g,
                    sign_g,
                    amp_factor_g,
                    trans_l,
                    signed_l,
                    abs_l,
                    n_branches_l,
                    eta_xyz,
                    n_steps,
                    group_size,
                    s_grid,
                    s1_idx,
                )
                result = row_from_aggregate(agg, group_size, time.perf_counter() - t0)
                group_rows.extend(_q_result_rows("gate", requested_total, actual_total, n_groups, group_size, group, result, s_grid))

            if "layer" in methods:
                t0 = time.perf_counter()
                agg = dp_q_curve_aggregate(
                    init_pauli,
                    trans_l,
                    signed_l,
                    abs_l,
                    n_branches_l,
                    eta_xyz,
                    n_steps,
                    group_size,
                    s_grid,
                    s1_idx,
                )
                result = row_from_aggregate(agg, group_size, time.perf_counter() - t0)
                group_rows.extend(_q_result_rows("layer", requested_total, actual_total, n_groups, group_size, group, result, s_grid))
    return group_rows


def _q_result_rows(method, requested_total, actual_total, n_groups, group_size, group, result, s_grid):
    rows = []
    for k, s in enumerate(s_grid):
        rows.append(
            {
                "method": method,
                "requested_total_samples": int(requested_total),
                "actual_total_samples": int(actual_total),
                "n_groups": int(n_groups),
                "samples_per_group": int(group_size),
                "group": int(group),
                "s": float(s),
                "q_plus": float(result["q_plus"][k]),
                "q_minus": float(result["q_minus"][k]),
                "plus_num": float(result["plus_num"][k]),
                "plus_den": float(result["plus_den"]),
                "minus_num": float(result["minus_num"][k]),
                "minus_den": float(result["minus_den"]),
                "runtime_s": float(result["runtime"]),
                "triggered_frac": float(result["triggered_frac"]),
            }
        )
    return rows

In [ ]:
q_group_rows = run_final_dp_q_groups()
q_summary = summarize_q_groups(q_group_rows)
q_summary[:4]

## Pooled, Group Mean, and MoM

For each sector and $s$, the three summaries are:

$$Q_{\rm pooled}=\frac{\sum_g N_g}{\sum_g D_g},$$

$$Q_{\rm group\ mean}=\frac{1}{G}\sum_g \frac{N_g}{D_g},$$

$$Q_{\rm MoM}=\operatorname{median}_g \frac{N_g}{D_g}.$$

If high-denominator groups have lower $Q_g$, pooled approaches from below while group mean/MoM approach from above. The `den_ess_groups` and `top4_den_share` columns diagnose denominator concentration.

In [ ]:
def plot_q_summary(q_summary, s_to_show=(1.0, 8.0)):
    methods = sorted({r["method"] for r in q_summary})
    sectors = ("plus", "minus")
    fig, axes = plt.subplots(len(methods), len(sectors), figsize=(10, 5.8), sharex=True)
    if len(methods) == 1:
        axes = np.array([axes])
    for i, method in enumerate(methods):
        for j, sector in enumerate(sectors):
            ax = axes[i, j]
            for s in s_to_show:
                rows = [r for r in q_summary if r["method"] == method and r["sector"] == sector and r["s"] == s]
                rows.sort(key=lambda r: r["actual_total_samples"])
                x = np.array([r["actual_total_samples"] for r in rows], dtype=float)
                ax.plot(x, [r["pooled_ratio"] for r in rows], marker="o", label=rf"pooled, s={s:g}")
                ax.plot(x, [r["mom_median_group_ratio"] for r in rows], marker="^", linestyle="--", label=rf"MoM, s={s:g}")
            ax.set_xscale("log")
            ax.set_title(f"{method} Q{sector[0]}")
            ax.set_xlabel("total samples")
            ax.grid(True, alpha=0.25)
            if j == 0:
                ax.set_ylabel("Q estimate")
            ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


plot_q_summary(q_summary);

## Command-Line Production Runs

For longer runs, prefer the resumable command-line scripts instead of running everything in the notebook.

Final-DP Q/MoM sweep:

```bash
MPLCONFIGDIR=.mplconfig python final_dp_mom_convergence_sweep.py \
  --n-qubits 20 --n-steps 10 --q1 6 --q2 10 --phi 0.2 \
  --s-grid 1,2,4,8 \
  --total-samples 1e3,3e3,1e4,3e4,1e5,3e5,1e6,3e6,1e7,3e7,1e8 \
  --n-groups 16 \
  --methods gate,layer \
  --out-prefix final_dp_mom_convergence_N20_L10_Z6Z10_phi02
```

MPS reference curve:

```bash
python sweep_mps_zizj_chi_vs_s.py \
  --n-qubits 20 --n-steps 10 --q1 6 --q2 10 --phi 0.2 \
  --s-values 0,1,2,4,8 \
  --chi-values 128,256,350 \
  --noise-model independent \
  --noise-placement layer \
  --out-csv mps_N20_L10_Z6Z10_phi02_chi_sweep.csv \
  --resume
```

Heavy-tail diagnostic after a Q sweep:

```bash
MPLCONFIGDIR=.mplconfig python diagnose_q_batch_heavytails.py \
  --batches final_dp_mom_convergence_N20_L10_Z6Z10_phi02_groups.csv \
  --samples-per-batch 6250000 \
  --out-prefix final_dp_mom_convergence_N20_L10_Z6Z10_phi02_taildiag_1e8
```

Here `--samples-per-batch` should match the group size for the total sample count being diagnosed. For 16 groups and total samples $10^8$, this is $6.25\times 10^6$.

## Interpretation Checklist

- If pooled, group mean, and MoM agree, the group size is large enough for this observable/depth.
- If pooled is below group mean/MoM, check denominator concentration with `den_ess_groups` and `top4_den_share`.
- If `den_ess_groups` is much smaller than the number of groups, the result is dominated by a few denominator-heavy prefix batches.
- The final-layer DP itself is exact conditional on the sampled prefix. Disagreement between pooled and MoM indicates prefix rare-event variance, not final-DP bias.
- For extrapolation, use the pooled-vs-MoM gap as a systematic diagnostic. MoM can stabilize finite-sample fits, but it can also suppress real rare-tail contributions if groups are too small.
- For MPS comparison, always check bond dimension / discarded weight before treating the curve as exact.

## Files To Add For GitHub

Core final-layer DP workflow:

```bash
git add \
  Pauli_path_Heis_full_layer_sampling_restricted.py \
  Pauli_path_Heis_mixture_trigger.py \
  benchmark_q_curve_convergence.py \
  final_dp_mom_convergence_sweep.py \
  notebooks/Final_Layer_DP_Usage.ipynb
```

Optional but useful diagnostics:

```bash
git add \
  compare_final_dp_vs_single_pauli_q.py \
  sweep_final_dp_vs_single_convergence.py \
  diagnose_q_batch_heavytails.py \
  benchmark_q_batchsize_convergence.py \
  qpm_numba_utils.py
```

Avoid adding generated `.csv`, `.png`, `.mplconfig`, `.ipynb_checkpoints`, `.DS_Store`, and `__pycache__` files unless the benchmark artifact itself is intentionally part of the release.